<a href="https://colab.research.google.com/github/icunicn/UAS/blob/main/Data_science3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data Understanding**
Data Understanding merupakan salah satu tahap dalam framework CRISP-DM untuk memperoleh knowledge dari suatu data untuk memecahkan masalah bisnis. Data understanding digunakan untuk eksplorasi data dan memeriksa distribusi data sekaligus mengidentifikasi ketidaknormalan data.

Pada tahap ini, analis berupaya mengenali struktur data, jenis atribut, sumber data, serta hubungan antar variabel yang berpotensi memengaruhi permasalahan bisnis yang dikaji. Pemahaman awal tersebut menjadi landasan penting untuk memastikan bahwa data yang digunakan relevan, representatif, dan mampu mendukung proses pengambilan keputusan.

## 1. Import Dataset

In [26]:
from google.colab import files
uploaded = files.upload()

In [80]:
!pip install xlrd
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

# Define color constants for consistent use across visualizations
WARNA_BIASA = '#378ADD'
WARNA_PRODUKSI = '#D85A30'

In [28]:
file_path = "/content/Tarikan bulanan RUMAH 2025 3(2).xlsx"
xls = pd.ExcelFile(file_path)
sheet_names = xls.sheet_names
print(sheet_names)

['JANUARI', 'FEBRUARI', 'MARET', 'APRIL', 'MEI', 'JUNI', 'JULI', 'AGUSTUS', 'SEPTEMBER']


## 2. Eksplorasi Dataset


### 2.1 Memeriksa kolom dan kesesuaian data untuk mempermudah memahami data

In [29]:
data_per_bulan = pd.read_excel(file_path, sheet_name=None)
data_per_bulan['JANUARI']

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,No,Nama,s.akhir,s.awal,Pakai,/meter,Total,Admin,Total akhir
3,1,NEMAT,2494,2471,23,500,11500,2000,13500
4,2,MATRAWI BAKSO,1788,1766,22,500,11000,2000,13000
...,...,...,...,...,...,...,...,...,...
963,961,TOKO AGUNG,269,260,9,500,4500,2000,6500
964,962,SLAMET MC,584,493,91,500,45500,2000,47500
965,963,SENEDI,1479,1273,206,500,103000,2000,105000
966,964,HERI ERMIN,785,783,2,500,1000,2000,3000


### 2.2 Memeriksa info dari setiap sheet supaya mudah dalam eleminiasi kolom yang tidak digunakan dalam penggabungan data

In [30]:
for nama_sheet, df in data_per_bulan.items():
    print(f"\nSheet: {nama_sheet}")
    print(df.columns)


Sheet: JANUARI
Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'],
      dtype='object')

Sheet: FEBRUARI
Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'],
      dtype='object')

Sheet: MARET
Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'],
      dtype='object')

Sheet: APRIL
Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'],
      dtype='object')

Sheet: MEI
Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8'],
      dtype='object')

Sheet: JUNI
Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4',
       'Unnamed: 5', '

inspeksi tipe data dari masing-masing kolom

In [81]:
type(df.dtypes), len(df.dtypes), df.dtypes

(pandas.core.series.Series,
 11,
 Bulan           object
 No               int64
 Nama            object
 s.akhir        float64
 s.awal           int64
 Pakai            int64
 /meter           int64
 Total            int64
 Admin            int64
 Total akhir    float64
 Segmen          object
 dtype: object)

### 2.3 Menggabungkan semua data sheet menjadi satu

In [31]:
xl = pd.ExcelFile(file_path)
cols = ['No', 'Nama', 's.akhir', 's.awal',
        'Pakai', '/meter', 'Total', 'Admin', 'Total akhir']
BULAN_ORDER = ['JANUARI','FEBRUARI','MARET','APRIL','MEI','JUNI',
                'JULI','AGUSTUS','SEPTEMBER']

all_dfs = []
for sheet in xl.sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet, header=3)
    df = df.dropna(axis=1, how='all')
    df = df[cols].copy()
    df = df.dropna(subset=['No'])
    df = df[pd.to_numeric(df['No'], errors='coerce').notna()]
    df['Bulan'] = sheet
    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
combined['No'] = combined['No'].astype(int)
combined = combined[['Bulan'] + cols]


print(f"Total baris: {len(combined)}")
display(combined.head(10))


Total baris: 8725


,Bulan,No,Nama,s.akhir,s.awal,Pakai,/meter,Total,Admin,Total akhir
0,JANUARI,1,NEMAT,2494.0,2471,23,500,11500,2000,13500.0
1,JANUARI,2,MATRAWI BAKSO,1788.0,1766,22,500,11000,2000,13000.0
2,JANUARI,3,SUBARI,388.0,388,0,500,0,2000,2000.0
3,JANUARI,4,JUMANTI,2780.0,2761,19,500,9500,2000,11500.0
4,JANUARI,5,JUKARI,497.0,491,6,500,3000,2000,5000.0
5,JANUARI,6,MUS,1057.0,1039,18,500,9000,2000,11000.0
6,JANUARI,7,JUMAK,181.0,177,4,500,2000,2000,4000.0
7,JANUARI,8,MESENI,35.0,35,0,500,0,2000,2000.0
8,JANUARI,9,SATUKIK,721.0,703,18,500,9000,2000,11000.0
9,JANUARI,10,KOSNADI,127.0,127,0,500,0,2000,2000.0


### 2.4 Memeriksa distribusi data di tabel

In [32]:
combined.describe()

,No,s.akhir,s.awal,Pakai,/meter,Total,Admin,Total akhir
count,8725.000000,8724.000000,8725.000000,8725.000000,8725.000000,8.725000e+03,8725.0,8.711000e+03
mean,485.224413,1603.330238,1573.460630,29.685845,511.575931,1.592407e+04,2000.0,1.811813e+04
std,279.874187,1474.539491,1451.583357,61.277634,75.197151,3.767078e+04,0.0,3.418050e+04
min,1.000000,0.000000,0.000000,-2935.000000,500.000000,-1.467500e+06,2000.0,2.000000e+03
25%,243.000000,620.000000,607.000000,7.000000,500.000000,3.500000e+03,2000.0,6.000000e+03
50%,485.000000,1287.000000,1263.000000,18.000000,500.000000,9.000000e+03,2000.0,1.100000e+04
75%,728.000000,2089.000000,2057.000000,35.000000,500.000000,1.750000e+04,2000.0,1.950000e+04
max,973.000000,14629.000000,14348.000000,1283.000000,1000.000000,1.005000e+06,2000.0,1.007000e+06


### 2.5 Data Preparation
Memeriksa apakah ada data yang tidak normal

In [105]:
combined = pd.concat(all_dfs, ignore_index=True)
combined['No'] = combined['No'].astype(int)
combined['Pakai'] = pd.to_numeric(combined['Pakai'], errors='coerce')
combined['Total akhir'] = pd.to_numeric(combined['Total akhir'], errors='coerce')
combined = combined[['Bulan'] + cols]

df = combined[combined['Pakai'] >= 0 ].copy()
# Add 'Segmen' column to df after it's created, for downstream analysis
df['Segmen'] = df['/meter'].map({500: 'Rumah Tangga Biasa', 1000: 'Rumah Tangga Produksi'})

combined['Segmen'] = combined['/meter'].map({500: 'Rumah Tangga Biasa', 1000: 'Rumah Tangga Produksi'})

print(f'Total baris (raw)  : {len(combined):,}')
print(f'Total baris (bersih): {len(df):,}')
print(f'Nilai negatif      : {(combined["Pakai"] < 0).sum()} baris (potensi data null)')
df.head()

Total baris (raw)  : 8,725
Total baris (bersih): 8,724
Nilai negatif      : 1 baris (potensi data null)


,Bulan,No,Nama,s.akhir,s.awal,Pakai,/meter,Total,Admin,Total akhir,Segmen
0,JANUARI,1,NEMAT,2494.0,2471,23,500,11500,2000,13500.0,Rumah Tangga Biasa
1,JANUARI,2,MATRAWI BAKSO,1788.0,1766,22,500,11000,2000,13000.0,Rumah Tangga Biasa
2,JANUARI,3,SUBARI,388.0,388,0,500,0,2000,2000.0,Rumah Tangga Biasa
3,JANUARI,4,JUMANTI,2780.0,2761,19,500,9500,2000,11500.0,Rumah Tangga Biasa
4,JANUARI,5,JUKARI,497.0,491,6,500,3000,2000,5000.0,Rumah Tangga Biasa


Periksa data yang bernilai null atau Nan

In [87]:
negatif = combined[combined['Pakai'] < 0][['Bulan','Nama','s.awal','s.akhir','Pakai']].sort_values('Pakai')
print(f'Total nilai negatif: {len(negatif)} baris')
print('Kemungkinan penyebab: stan awal diinput lebih besar dari stan akhir (salah catat)\n')
print(negatif.to_string(index=False))

Total nilai negatif: 1 baris
Kemungkinan penyebab: stan awal diinput lebih besar dari stan akhir (salah catat)

   Bulan      Nama  s.awal  s.akhir  Pakai
FEBRUARI TATIK ROI    2935      NaN  -2935


Terdapat missing value dari data dibulan Februari. Replace data s.akhir == s.awal supaya data dapat di process

In [88]:
# set s.akhir = s.awal untuk baris dengan nilai Pakai negatif
combined.loc[combined['Pakai'] < 0, 's.akhir'] = combined['s.awal']

Cek apakah missing value masih ada.

In [90]:
negatif = combined[combined['Pakai'] < 0][
    ['Bulan','Nama','s.awal','s.akhir','Pakai']
].sort_values('Pakai')
combined['Pakai'] = combined['s.akhir'] - combined['s.awal']
print(f'Total nilai negatif: {len(negatif)} baris')
print(negatif.to_string(index=False))

Total nilai negatif: 0 baris
Empty DataFrame
Columns: [Bulan, Nama, s.awal, s.akhir, Pakai]
Index: []


Periksa kolom tarif apakah semua memiliki value yang sama

In [37]:
combined['Segmen'] = combined['/meter'].map({500: 'Rumah Tangga Biasa', 1000: 'Rumah Tangga Produksi'})
print(combined['/meter'].value_counts())

/meter
500     8523
1000     202
Name: count, dtype: int64


## Analisis Data Lanjutan

Distribusi data berdasarkan statistik deskripsi

In [93]:
def iqr_outliers(x):
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    iqr = q3 - q1
    return ((x < q1 - 1.5*iqr) | (x > q3 + 1.5*iqr)).sum()

stats = df.groupby('Bulan')['Pakai'].agg(
    n='count',
    mean='mean',
    median='median',
    std='std',
    min='min',
    max='max',
    q25=lambda x: x.quantile(0.25),
    q75=lambda x: x.quantile(0.75),
).round(1)

stats['outliers']  = df.groupby('Bulan')['Pakai'].apply(iqr_outliers).values

stats = stats.reindex(BULAN_ORDER)
stats.index.name = 'Bulan'

print('Statistik Deskripsi Pemakaian Air per Bulan (m³):')
print(stats.to_string())

Statistik Deskripsi Pemakaian Air per Bulan (m³):
             n  mean  median   std  min   max   q25   q75  outliers
Bulan                                                              
JANUARI    965  37.7    23.0  61.3    0   926  10.0  43.0        85
FEBRUARI   968  25.6    16.0  52.3    0  1283   6.0  30.0        65
MARET      969  27.2    18.0  34.5    0   365   8.0  33.0        80
APRIL      969  35.5    23.0  49.6    0   518  10.0  41.0        81
MEI        969  35.7    20.0  80.6    0  1061   9.0  36.0        83
JUNI       969  20.1    11.0  36.9    0   620   5.0  21.0        87
JULI       971  20.0    11.0  45.2    0  1005   5.0  21.0        79
AGUSTUS    971  37.8    25.0  49.4    0   472  11.0  44.0        70
SEPTEMBER  973  30.8    21.0  43.2    0   491   9.0  37.0        61


Rata - rata dan median data tiap bulan

In [50]:
fig = go.Figure()

fig.add_trace(go.Bar(
    x=BULAN_ORDER, y=stats['mean'],
    name='Rata-rata', marker_color='#378ADD', opacity=0.85
))
fig.add_trace(go.Scatter(
    x=BULAN_ORDER, y=stats['median'],
    name='Median', mode='lines+markers',
    line=dict(color='#D85A30', width=2.5),
    marker=dict(size=8)
))

fig.update_layout(
    title='Rata-rata vs Median Pemakaian Air per Bulan',
    xaxis_title='Bulan', yaxis_title='Pemakaian (m³)',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    plot_bgcolor='white', height=420,
    xaxis=dict(tickangle=-30)
)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

Distibusi segmen tarif pemakaian seluruh bulan

In [95]:
bins   = [-0.01, 0.01, 10, 30, 50, 100, 200, 99999]
labels = ['Nol','1-10','11-30','31-50','51-100','101-200','201+']

combined['bucket'] = pd.cut(combined['Pakai'], bins=bins, labels=labels)
dist = combined['bucket'].value_counts().reindex(labels)
pct  = (dist / dist.sum() * 100).round(1)

colors = ['#9ca3af','#60a5fa','#34d399','#fbbf24','#f97316','#ef4444','#8b5cf6']
fig = go.Figure(go.Bar(
    x=labels, y=dist.values,
    text=[f'{v:,}<br>({p}%)' for v, p in zip(dist.values, pct.values)],
    textposition='outside',
    marker_color=colors
))
fig.update_layout(
    title='Distribusi Segmen Pemakaian (seluruh bulan)',
    xaxis_title='Segmen Pemakaian (m³)',
    yaxis_title='Jumlah Rekening',
    plot_bgcolor='white', height=430,
    uniformtext_minsize=10, uniformtext_mode='hide'
)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

Rata-rata pemakaian bulanan per segmen tarif

In [106]:
combined['Segmen'] = combined['/meter'].map({500: 'Rumah Tangga Biasa', 1000: 'Rumah Tangga Produksi'})
avg_bulan = combined.groupby(['Bulan','Segmen'], observed=True)['Pakai'].mean().reset_index().sort_values('Bulan')

fig = go.Figure()
for seg, color in [('Rumah Tangga Biasa', WARNA_BIASA),
                    ('Rumah Tangga Produksi', WARNA_PRODUKSI)]:
    sub = avg_bulan[avg_bulan['Segmen']==seg]
    fig.add_trace(go.Scatter(
        x=sub['Bulan'].astype(str), y=sub['Pakai'].round(1),
        name=seg, mode='lines+markers',
        line=dict(color=color, width=2.5),
        marker=dict(size=8)
    ))

fig.update_layout(
    title='Rata-rata Pemakaian Bulanan per Segmen Tarif',
    yaxis_title='Rata-rata Pemakaian (m³)',
    plot_bgcolor='white', height=430,
    xaxis=dict(tickangle=-30), legend=dict(orientation='h', y=1.08)
)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

In [108]:
print('=== 3 PEMAKAIAN TERTINGGI PER BULAN ===')
for b in BULAN_ORDER:
    sub  = combined[combined['Bulan'] == b]
    q1, q3 = sub['Pakai'].quantile(0.25), sub['Pakai'].quantile(0.75)
    iqr  = q3 - q1
    batas = q3 + 1.5 * iqr
    outliers = sub[sub['Pakai'] > batas].nlargest(3, 'Pakai')[['Nama','Pakai']]
    print(f'\n{b} (batas atas: {batas:.0f} m³):')
    for _, r in outliers.iterrows():
        print(f'  {r["Nama"]:<30} {r["Pakai"]:>6.0f} m³')

=== 3 PEMAKAIAN TERTINGGI PER BULAN ===

JANUARI (batas atas: 92 m³):
  LAKSONO                           926 m³
  UNGGUL KEDAI BAMBU                720 m³
  HARI NUR                          680 m³

FEBRUARI (batas atas: 66 m³):
  UST SOLIKIN                      1283 m³
  HARI NUR                          503 m³
  DOL BESUKI                        303 m³

MARET (batas atas: 70 m³):
  UST SOLIKIN                       365 m³
  BONI PONDOK TUBING                331 m³
  BUANI                             267 m³

APRIL (batas atas: 88 m³):
  HJ IS                             518 m³
  SAROH ILA                         497 m³
  UST SOLIKIN                       450 m³

MEI (batas atas: 76 m³):
  P HARDI                          1061 m³
  PONIRI                           1031 m³
  HJ IS                             977 m³

JUNI (batas atas: 45 m³):
  KUJIT UMI                         620 m³
  NUSA PELANGI                      436 m³
  INTAYU                            322 m³

JULI (batas ata

In [58]:
print('=== Deteksi Outlier per Segmen Tarif ===')
for seg in ['Rumah Tangga Biasa', 'Rumah Tangga Produksi']:
    sub  = combined[combined['Segmen']==seg]['Pakai']
    q1, q3 = sub.quantile(0.25), sub.quantile(0.75)
    batas = q3 + 1.5*(q3-q1)
    out   = combined[(combined['Segmen']==seg) & (combined['Pakai'] > batas)]
    print(f'\n{seg}')
    print(f'  Q1={q1:.0f}, Q3={q3:.0f}, Batas atas={batas:.0f} m³')
    print(f'  Outlier: {len(out)} dari {len(combined[combined["Segmen"]==seg])} rekening ({len(out)/len(combined[combined["Segmen"]==seg])*100:.1f}%)')
    print(f'  Top 5:')
    top5 = out.nlargest(5,'Pakai')[['Bulan','Nama','Pakai','Total akhir']]
    print(top5.to_string(index=False))

=== Deteksi Outlier per Segmen ===

Rumah Tangga Biasa
  Q1=7, Q3=34, Batas atas=74 m³
  Outlier: 620 dari 8523 rekening (7.3%)
  Top 5:
   Bulan        Nama  Pakai  Total akhir
FEBRUARI UST SOLIKIN   1283     643500.0
     MEI     P HARDI   1061     532500.0
     MEI      PONIRI   1031     517500.0
     MEI       HJ IS    977     490500.0
     MEI      JAINUL    934     469000.0

Rumah Tangga Produksi
  Q1=7, Q3=129, Batas atas=312 m³
  Outlier: 17 dari 202 rekening (8.4%)
  Top 5:
  Bulan               Nama  Pakai  Total akhir
   JULI MOHTAR GREEN HOUSE   1005    1007000.0
    MEI        REST AREA 1    779     781000.0
JANUARI UNGGUL KEDAI BAMBU    720     722000.0
AGUSTUS       NUSA PELANGI    472     474000.0
    MEI             INTAYU    454     456000.0


In [59]:
zero_pct = combined.groupby('Bulan')['Pakai'].apply(lambda x: (x==0).sum() / len(x) * 100).reindex(BULAN_ORDER).round(1)

fig = go.Figure(go.Bar(
    x=zero_pct.index, y=zero_pct.values,
    text=[f'{v}%' for v in zero_pct.values],
    textposition='outside',
    marker_color='#94a3b8'
))
fig.update_layout(
    title='Persentase Pelanggan dengan Pemakaian Nol per Bulan',
    xaxis_title='Bulan', yaxis_title='% Pelanggan',
    plot_bgcolor='white', height=400,
    xaxis=dict(tickangle=-30)
)
fig.update_yaxes(gridcolor='#f0f0f0', range=[0, 20])
fig.show()

In [112]:
tarif_unik = combined.groupby('Nama')['/meter'].nunique()
# cek apakah ada pelanggan dengan lebih dari 1 tarif
ganti_tarif = (tarif_unik > 1).any()

print(ganti_tarif)

True


In [113]:
tarif_unik = combined.groupby('Nama')['/meter'].nunique()
ganti_tarif = tarif_unik[tarif_unik > 1].index.tolist()

print(f'=== Pelanggan Berganti Segmen Tarif ===')
print(f'Jumlah: {len(ganti_tarif)} pelanggan\n')
for nama in ganti_tarif:
    detail = combined[combined['Nama']==nama][['Bulan','Pakai','/meter','Segmen','Total akhir']]
    print(f'--- {nama} ---')
    print(detail.to_string(index=False))
    print()

=== Pelanggan Berganti Segmen Tarif ===
Jumlah: 3 pelanggan

--- NUSA PELANGI 2 ---
    Bulan  Pakai  /meter                Segmen  Total akhir
     JULI    174     500    Rumah Tangga Biasa      89000.0
  AGUSTUS     21     500    Rumah Tangga Biasa      12500.0
SEPTEMBER     22    1000 Rumah Tangga Produksi      24000.0

--- PUSPA HOMESTAY ---
    Bulan  Pakai  /meter                Segmen  Total akhir
  JANUARI     14    1000 Rumah Tangga Produksi      16000.0
 FEBRUARI      0    1000 Rumah Tangga Produksi       2000.0
    MARET      8     500    Rumah Tangga Biasa       6000.0
    APRIL     12     500    Rumah Tangga Biasa       8000.0
      MEI     23     500    Rumah Tangga Biasa      13500.0
     JUNI     21     500    Rumah Tangga Biasa      12500.0
     JULI      6     500    Rumah Tangga Biasa       5000.0
  AGUSTUS    103     500    Rumah Tangga Biasa      53500.0
SEPTEMBER      0     500    Rumah Tangga Biasa       2000.0

--- SATUMAN KEBUN ---
    Bulan  Pakai  /meter     

Ditemukan beberapa pelanggan yang mengalami perubahan segmen tarif

# Bussiness Understanding

Korelasi data terhadap bisnis HIPAM


## 1. Total pendapatan akhir berdasarkan segmen

In [65]:
rev = combined.groupby('Segmen')['Total akhir'].sum()
n_pel = combined.groupby('Segmen')['Nama'].nunique()
total_rev = rev.sum()

print('=== Revenue Mix ===')
for seg in rev.index:
    arpu = rev[seg] / n_pel[seg]
    print(f'{seg}')
    print(f'  Pelanggan  : {n_pel[seg]}')
    print(f'  Revenue    : Rp {rev[seg]:>15,.0f}  ({rev[seg]/total_rev*100:.1f}%)')
    print()
print(f'Total Revenue Keseluruhan: Rp {total_rev:,.0f}')

# Pie chart
fig = go.Figure(go.Pie(
    labels=rev.index.tolist(),
    values=rev.values,
    hole=0.45,
    marker_colors=[WARNA_BIASA, WARNA_PRODUKSI],
    textinfo='label+percent+value',
    texttemplate='%{label}<br>Rp %{value:,.0f}<br>(%{percent})'
))
fig.update_layout(
    title='Komposisi Revenue: Biasa vs Produksi',
    height=420, showlegend=False
)
fig.show()

=== Revenue Mix ===
Rumah Tangga Biasa
  Pelanggan  : 857
  Revenue    : Rp     138,557,000  (87.8%)

Rumah Tangga Produksi
  Pelanggan  : 25
  Revenue    : Rp      19,270,000  (12.2%)

Total Revenue Keseluruhan: Rp 157,827,000


## 2. Total Pendapatan Bulanan

In [74]:
rev_bulan_total = (
    combined
    .groupby('Bulan', observed=True)['Total akhir']
    .sum()
    .reset_index()
)

rev_bulan_total['Bulan'] = rev_bulan_total['Bulan'].astype(str)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=rev_bulan_total['Bulan'],
    y=rev_bulan_total['Total akhir'],
    name='Total Revenue',
))

fig.update_layout(
    title='Total Revenue Bulanan',
    yaxis_title='Revenue (Rp)',
    plot_bgcolor='white',
    height=430,
    xaxis=dict(tickangle=-30),
    yaxis_tickformat=',.0f'
)

fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

## 3. Total Pendapatan Bulanan per segmen

In [66]:
rev_bulan = combined.groupby(['Bulan','Segmen'], observed=True)['Total akhir'].sum().reset_index()
rev_bulan['Bulan'] = rev_bulan['Bulan'].astype(str)

fig = go.Figure()
for seg, color in [('Rumah Tangga Biasa', WARNA_BIASA),
                    ('Rumah Tangga Produksi', WARNA_PRODUKSI)]:
    sub = rev_bulan[rev_bulan['Segmen']==seg]
    fig.add_trace(go.Bar(
        x=sub['Bulan'], y=sub['Total akhir'],
        name=seg, marker_color=color
    ))

fig.update_layout(
    title='Revenue Bulanan per Segmen Tarif',
    barmode='stack', yaxis_title='Revenue (Rp)',
    plot_bgcolor='white', height=430,
    xaxis=dict(tickangle=-30),
    legend=dict(orientation='h', y=1.08),
    yaxis_tickformat=',.0f'
)
fig.update_yaxes(gridcolor='#f0f0f0')
fig.show()

## 4. Rata-rata pendapatan per orang (ARPU)

In [115]:
n_bulan = combined['Bulan'].nunique()

total_pel_biasa = combined[combined['Segmen'] == 'Rumah Tangga Biasa']['Nama'].nunique()
total_pel_prod  = combined[combined['Segmen'] == 'Rumah Tangga Produksi']['Nama'].nunique()

rev_biasa  = combined[combined['Segmen'] == 'Rumah Tangga Biasa']['Total akhir'].sum()
rev_prod   = combined[combined['Segmen'] == 'Rumah Tangga Produksi']['Total akhir'].sum()
total_rev  = combined['Total akhir'].sum()
total_vol  = combined['Pakai'].sum()

# Hitung ARPU (Average Revenue Per User) per bulan
arpu_biasa = rev_biasa / total_pel_biasa / n_bulan
arpu_prod  = rev_prod  / total_pel_prod  / n_bulan

# Hitung persentase revenue
pct_biasa = rev_biasa / total_rev * 100
pct_prod  = rev_prod  / total_rev * 100

# Cetak ringkasan
print('=' * 50)
print('  RINGKASAN EKSEKUTIF - TARIKAN AIR RUMAH 2025')
print('=' * 50)


print(f'Total rekening        : {len(combined):,}')
print(f'Pelanggan Biasa       : {total_pel_biasa:,} orang')
print(f'Pelanggan Produksi    : {total_pel_prod:,} orang')
print(f'Total volume tercatat : {total_vol:,.0f} m³')

print('\n--- PENDAPATAN ---')
print(f'Total revenue         : Rp {total_rev:,.0f}')
print(f'Revenue Biasa         : Rp {rev_biasa:,.0f} ({pct_biasa:.1f}%)')
print(f'Revenue Produksi      : Rp {rev_prod:,.0f} ({pct_prod:.1f}%)')

print('\n--- ARPU PER BULAN ---')
print(f'ARPU Biasa/bulan      : Rp {arpu_biasa:,.0f}')
print(f'ARPU Produksi/bulan   : Rp {arpu_prod:,.0f}')
print(f'Selisih               : {arpu_prod / arpu_biasa:.1f}x lebih tinggi')

print('\n' + '=' * 50)

  RINGKASAN EKSEKUTIF - TARIKAN AIR RUMAH 2025
Total rekening        : 8,725
Pelanggan Biasa       : 857 orang
Pelanggan Produksi    : 25 orang
Total volume tercatat : 259,009 m³

--- PENDAPATAN ---
Total revenue         : Rp 157,827,000
Revenue Biasa         : Rp 138,557,000 (87.8%)
Revenue Produksi      : Rp 19,270,000 (12.2%)

--- ARPU PER BULAN ---
ARPU Biasa/bulan      : Rp 17,964
ARPU Produksi/bulan   : Rp 85,644
Selisih               : 4.8x lebih tinggi



Dari data tersebut, ARPU produksi 4.8 kali lebih banyak daripada rumah tangga biasa. Artinya segmen produksi menjadi **penyumbang pendapatan utama** meskipun jumlah pelanggannya lebih sedikit